# nanochat.cpp data parallel on two T4 cards (Kaggle)

Run the best-fit nanochat model with data parallel on two T4 cards in one
Kaggle session.

One process owns each card. The two processes average their gradients once per
step, so the parameter update equals a one-card run.

This notebook uses the host reference gradient sync in `src/distributed.cc`.
The sync sends each gradient over loopback TCP and needs no NCCL. The CUDA
backend stages each gradient through host memory, because the model lives in
device memory. The design is `docs/distributed-design.md`.

NCCL is the later fast path (phase C3 in `docs/distributed-t4-plan.md`).

**Before you run this notebook**, push a revision that carries the staging
change and the `notebooks/parallel.py` helper. Then set `CODE_REF` in the next
cell to that branch or commit.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

# The Hugging Face token lets the data download run without a prompt.
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret; the download may be rate limited")

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")

# The revision that carries the two-card staging change and parallel.py.
CODE_REF = None

# Clone the repository, then make its packages importable.
if not REPO.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
sys.path[:0] = [str(REPO / "python"), str(REPO / "notebooks"),
                str(REPO / ".agents" / "skills" / "model-fit" / "scripts")]
print("repository:", REPO)

## Setup

`kaggle_setup.setup` fetches the repository, runs `tools/kaggle/bootstrap.sh`,
loads the host environment, and builds the T4 shared library. The lower cells
then build the standalone `train_main` binary for the two ranks.

In [ ]:
import kaggle_setup

host = kaggle_setup.setup(url=REPO_URL, dest=REPO, ref=CODE_REF, arch="sm_75")
print("library:", host.library)
print("build key:", host.build_key)

devices = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total",
     "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(devices)
assert len(devices.splitlines()) == 2, "this notebook needs two T4 cards"

In [ ]:
from pi_agent import run_pi

PI_PROMPT = (
    "Inspect this repository and report a concise status. "
    "Do not edit files.")
run_pi(PI_PROMPT, cwd=REPO)

## Build the two-card binary

`train_main` is a separate target from the shared library. Build it with the
complete Turing config, `--config=t4` (`sm_75`, fp32, CUDA backend).

In [ ]:
build = subprocess.run(
    ["tools/nanochat", "build", "//src:train_main", "--config=t4"],
    cwd=str(REPO), capture_output=True, text=True)
print(build.stdout[-2000:])
assert build.returncode == 0, build.stderr[-4000:]

TRAIN_MAIN = REPO / "bazel-bin" / "src" / "train_main"
assert TRAIN_MAIN.is_file(), TRAIN_MAIN
print("train_main:", TRAIN_MAIN)

## Data and tokenizer

One ClimbMix shard trains the tokenizer and the model. The validation shard is
the reference split. Both ranks read the same files and shard documents by a
stride.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub"], check=True)

from huggingface_hub import snapshot_download

DATA = BASE / "base_data_climbmix"
TRAIN_SHARD = "shard_00000.parquet"
VAL_SHARD = "shard_06542.parquet"

if not (DATA / VAL_SHARD).is_file():
    snapshot_download(repo_id="karpathy/climbmix-400b-shuffle",
                      repo_type="dataset", local_dir=str(DATA),
                      allow_patterns=[TRAIN_SHARD, VAL_SHARD],
                      token=os.environ.get("HF_TOKEN"))

print("train:", DATA / TRAIN_SHARD, "validation:", DATA / VAL_SHARD)

In [ ]:
import nanochat_cpp as nc

TOKENIZER = BASE / "tokenizer" / "tokenizer.nctoken"

if not TOKENIZER.is_file():
    TOKENIZER.parent.mkdir(parents=True, exist_ok=True)
    nc.toolchain.build(["//src/tokenizer:tok_train_main"])
    nc.toolchain.run("t0-cpu", [
        "bazel-bin/src/tokenizer/tok_train_main",
        "--parquet", f"{DATA}/{TRAIN_SHARD}",
        "--vocab-size", "32768",
        "--out", str(TOKENIZER),
    ])

tokenizer = nc.Tokenizer.load(str(TOKENIZER))
assert tokenizer.decode(tokenizer.encode("The capital of France is")) == \
    "The capital of France is", "the tokenizer round trip failed"
print("tokenizer:", tokenizer.vocab_size, "tokens")

## Smoke run

A small model proves the reduce before the long run. The model has four layers
and a hidden size of 256. Each rank consumes four micro-batches of 16 sequences, each of 512
tokens. The cell starts one process per card and waits.

The helper sets `CUDA_VISIBLE_DEVICES` to the rank, so each process sees one
card as device 0. Rank 0 binds the port. Rank 1 connects.

In [ ]:
import parallel

RUNS = Path("/kaggle/working/runs")

smoke = parallel.ParallelPlan(
    train_parquet=f"{DATA}/{TRAIN_SHARD}",
    val_parquet=f"{DATA}/{VAL_SHARD}",
    tokenizer=str(TOKENIZER),
    out_dir=str(RUNS / "smoke-2xt4"),
    batch=16, grad_accum=4, num_iterations=20,
    preset=None, flags=parallel.SMOKE_FLAGS,
    log_every=5, save_every=20, seed=7)

print(" ".join(parallel.command_for(0, smoke)))
result = parallel.launch(smoke, root=REPO, timeout=1800)
parallel.assert_complete(result)
print("returncodes:", result.returncodes)

### Smoke check

Each rank logs the loss over its own shard, so the two curves differ. The
parameters and the checkpoints must be equal, because the reduced gradient is
equal. The two checkpoint hashes below must match.

In [ ]:
import hashlib

import matplotlib.pyplot as plt
import pandas as pd

frame = pd.DataFrame(parallel.loss_rows(result))
print(frame)


def sha256(path):
    path = Path(path)
    return hashlib.sha256(path.read_bytes()).hexdigest() \
        if path.is_file() else "missing"


digests = {rank: sha256(smoke.checkpoint_path(rank))
           for rank in range(smoke.world_size)}
print(digests)
assert digests[0] == digests[1] != "missing", \
    "the two rank checkpoints differ: the gradient reduce is broken"

frame.plot(x="step", y=["rank0", "rank1"])
plt.title("two-card smoke run")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

## Best-fit run

The `model-fit` skill picks the shape from the source formulas. The precision is fp32.
The `track3` preset disables the value embeddings and sets the optimizer and the
schedule. The shape flags override the baseline shape.

The best fit is 16 layers, hidden 1024, 8 heads, 4 key/value heads, and 4096
tokens. It needs about 9.1 GiB of the 16 GiB card. See
`docs/distributed-t4-plan.md` section 3.

`num_iterations` bounds the run. Keep the value small for a first Kaggle
session. Raise it after the smoke run proves the reduce.

In [ ]:
import model_fit

best_fit = parallel.ParallelPlan(
    train_parquet=f"{DATA}/{TRAIN_SHARD}",
    val_parquet=f"{DATA}/{VAL_SHARD}",
    tokenizer=str(TOKENIZER),
    out_dir=str(RUNS / "best-fit-2xt4"),
    batch=1, grad_accum=64, num_iterations=100,
    preset="track3",
    flags=model_fit.shape_flags(model_fit.BEST_FIT_T4),
    log_every=1, eval_every=10, eval_steps=8, save_every=10, seed=42)

print(" ".join(parallel.command_for(0, best_fit)))
result3 = parallel.launch(best_fit, root=REPO, timeout=6 * 3600)
parallel.assert_complete(result3)
print("returncodes:", result3.returncodes)

In [ ]:
frame3 = pd.DataFrame(parallel.loss_rows(result3))
print(frame3.tail())
frame3.plot(x="step", y=["rank0", "rank1"])
plt.title("best fit on two T4 cards")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

for rank in range(best_fit.world_size):
    path = best_fit.checkpoint_path(rank)
    size = path.stat().st_size if path.is_file() else "missing"
    print("rank", rank, path, size)

## Notes

- The two loss curves differ by design. Each rank logs the mean loss over its
  own document shard. The reduced gradient, the parameters, and the checkpoints
  are equal.
- The run directories are under `/kaggle/working/runs`. Keep the notebook
  version to keep the outputs.
- The host reference makes one TCP round trip per parameter per step. It is the
  correctness path, not the speed path.
- The `model-fit` skill computes the shape and the memory from the source
  formulas. Set `batch` and `grad_accum` from its output.
- `train.cc` reports `tok/s` and `mfu` for one micro-batch, so the two values
  read low by a factor of `grad_accum`. The loss does not change.
- To add NCCL later, write `backends/cuda/nccl_sync.cu` and replace the sync.
  See `docs/distributed-t4-plan.md` phase C3.
- A resumed run restarts the epoch, because the checkpoint does not store the
  document index in a later phase.